# FruitVision: treinamento no Google Colab

Antes de rodar:

1. No Colab, vá em **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.
2. Envie o arquivo `fruitvision_colab.zip` (gerado na raiz do projeto) para a pasta **MyDrive** do seu Google Drive.
3. Rode as células na ordem.

O modelo é treinado **do zero** (`yolo26n.yaml`), sem pesos pré-treinados e sem nenhuma API de IA externa.

## 1. Montar o Google Drive e extrair o projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_NO_DRIVE = '/content/drive/MyDrive/fruitvision_colab.zip'
!rm -rf /content/fruitvision && mkdir -p /content/fruitvision
!unzip -q "$ZIP_NO_DRIVE" -d /content/fruitvision
%cd /content/fruitvision
!ls

## 2. Instalar dependências e conferir a GPU

In [ ]:
!pip install -q ultralytics
import torch
print('GPU disponível:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA (troque o ambiente para GPU!)')

## 3. Auditoria do dataset

In [ ]:
!python training/audit_dataset.py

## 4. Treinamento do zero

Com a GPU T4, 100 épocas devem levar algo em torno de 20–40 minutos. O `patience=20` para antes se o modelo parar de melhorar.
Como a GPU tem memória de sobra, dá para usar `--batch 16`.

In [ ]:
!python training/train.py --epochs 100 --batch 16

## 5. Validação (métricas para o relatório)

In [ ]:
!python training/validate.py

## 6. Gráficos do treino

In [ ]:
from IPython.display import Image, display
RUN = 'runs/fruitvision_yolo26n_scratch'
for img in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'val_batch0_pred.jpg']:
    try:
        print(img); display(Image(f'{RUN}/{img}', width=900))
    except Exception as e:
        print('  (não encontrado)', e)

## 7. Salvar os resultados no Drive

Gera `fruitvision_runs.zip` no seu Drive. Baixe e extraia na **raiz do projeto**; vai aparecer a pasta
`runs/fruitvision_yolo26n_scratch/weights/best.pt`, que o backend usa.

In [ ]:
!zip -qr fruitvision_runs.zip runs
!cp fruitvision_runs.zip /content/drive/MyDrive/
print('Salvo em MyDrive/fruitvision_runs.zip')

## 8. (Opcional) Comparação com transfer learning

**Só rode se o professor permitir** o uso de pesos pré-treinados (o README diz que o treino do zero é a primeira etapa).
Serve para comparar no relatório: do zero vs. pré-treinado.

In [ ]:
!python training/train.py --pretrained --epochs 100 --batch 16
!python training/validate.py --weights runs/fruitvision_yolo26n_pretrained/weights/best.pt
!zip -qr fruitvision_runs.zip runs && cp fruitvision_runs.zip /content/drive/MyDrive/